In [3]:
from pathlib import Path
print(Path.cwd())
print((Path.cwd() / "helpers.py").exists())

/files
False


In [7]:
import sys
from pathlib import Path

racine = Path.cwd()
trouve = [p.parent for p in racine.rglob("helpers.py") if "ada" not in p.parts]
if trouve:
    sys.path.insert(0, str(trouve[0]))
print(trouve)

[PosixPath('/files/week-03/Project/code')]


In [8]:
from helpers import *
import pandas_datareader.data as web

series = ["GS10", "TB3MS", "BAA10YM", "UNRATE", "CPIAUCSL", "INDPRO"]
fred = web.DataReader(series, "fred", start="1950-01-01")
fred.to_csv(DATA / "fred_raw.csv")

macro = pd.DataFrame(index=fred.index)
macro["slope"] = fred["GS10"] - fred["TB3MS"]
macro["credit_spread"] = fred["BAA10YM"]
macro["unemployment"] = fred["UNRATE"]
macro["inflation"] = fred["CPIAUCSL"].pct_change(12, fill_method=None) * 100
macro["indpro_growth"] = fred["INDPRO"].pct_change(12, fill_method=None) * 100

n_avant = len(macro)
macro = macro.dropna()
print(n_avant, "mois avant nettoyage,", len(macro), "après")
print(macro.index.min().date(), macro.index.max().date())
macro.to_csv(DATA / "macro_monthly.csv")

922 mois avant nettoyage, 881 après
1953-04-01 2026-09-01


In [9]:
ff5 = web.DataReader("F-F_Research_Data_5_Factors_2x3", "famafrench", start="1950-01-01")
mom = web.DataReader("F-F_Momentum_Factor", "famafrench", start="1950-01-01")

ff5_m = ff5[0]
mom_m = mom[0]
mom_m.columns = [c.strip() for c in mom_m.columns]

factors = ff5_m[["SMB", "HML", "RMW", "RF"]].join(mom_m["Mom"], how="inner")
factors = factors.rename(columns={"SMB": "size", "HML": "value", "RMW": "quality", "RF": "rf", "Mom": "momentum"})
factors.index = factors.index.to_timestamp()

print(factors.index.min().date(), factors.index.max().date(), factors.shape)
factors.to_csv(DATA / "factors_monthly.csv")

1963-07-01 2026-09-01 (759, 5)
